# Notebook 10 — M2 Climate-Augmented Logistic Regression

**Purpose**: Fit the M2 logit by adding climate variables (`hurricane_hit_in_year`, `hurricane_frequency_5yr`) to the M1 specification. Compare M1 vs M2 to quantify the incremental predictive contribution of climate exposure.

**Inputs**
- `fl_model_ready.parquet` (from Notebook 08)
- `m1_logit.pkl`, `m1_metrics.csv` (from Notebook 09) — for direct comparison

**Outputs**
- `m2_logit.pkl` — fitted model
- `m2_coefficients.csv` — coefficients, odds ratios, CIs
- `m2_metrics.csv` — AUC and Brier score on train / test
- `m1_vs_m2_comparison.csv` — head-to-head comparison table

**Key comparison metrics**
- **ΔAUC (test)**: primary evidence of incremental discrimination
- **ΔBrier (test)**: incremental calibration improvement
- **Likelihood ratio test**: joint significance of the two climate variables
- **Coefficient stability**: does adding climate change M1 coefficients (especially ORIG_RATE)?


In [21]:
# Find any leftover old variable names in this notebook
import re
old_names = ['hurricane_hit_in_year', 'hurricane_frequency_5yr']

# Read the notebook file itself
import json
from pathlib import Path

nb_path = Path("your/data/path/here")

with open(nb_path, 'r', encoding='utf-8') as f:
    nb = json.load(f)

for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] != 'code':
        continue
    src = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
    for old in old_names:
        if old in src:
            print(f"⚠️  Cell {i} still contains '{old}'")

print("\nDone.")

⚠️  Cell 21 still contains 'hurricane_hit_in_year'
⚠️  Cell 21 still contains 'hurricane_frequency_5yr'

Done.


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss
from scipy import stats
import joblib

# TODO: update these to your paths
DATA_DIR = Path("your/data/path/here")
M1_DIR   = Path("your/data/path/here")
OUT_DIR  = Path("your/data/path/here")
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load model-ready data and M1 artifacts

In [3]:
df = pd.read_parquet(DATA_DIR / "fl_model_ready.parquet")
m1_model   = joblib.load(M1_DIR / "m1_logit.pkl")
m1_metrics = pd.read_csv(M1_DIR / "m1_metrics.csv")

print(f"Rows: {len(df):,}")
print(f"Default rate: {df['default_180dpd'].mean():.4f}")
print(f"\nM1 metrics (for reference):")
print(m1_metrics)


Rows: 19,625
Default rate: 0.0285

M1 metrics (for reference):
   split      n       auc     brier
0  train  13737  0.785048  0.026724
1   test   5888  0.766435  0.026743


## 2. Define M2 feature set = M1 + climate

In [22]:
# Same helper columns as M1 (must match Notebook 09)
df['NO_UNITS_GRP'] = np.where(df['NO_UNITS'] == 1, '1', '2plus')
df['HAS_MI']       = (df['MI_PCT'] > 0).astype(int)

numeric_features = [
    'CSCORE_B',
    'DTI',
    'ORIG_CLTV',
    'ORIG_RATE',
    'ORIG_UPB',
    'ORIG_TERM',
    'NUM_BORR',
    'MI_PCT',
    'HAS_MI',
    # --- NEW: climate variables ---
    'hurricanes_perf_window',
]

categorical_features = [
    'FIRST_FLAG',
    'OCC_STAT',
    'PROP',
    'CHANNEL',
    'NO_UNI
M2 numeric features    (10): ['CSCORE_B', 'DTI', 'ORIG_CLTV', 'ORIG_RATE', 'ORIG_UPB', 'ORIG_TERM', 'NUM_BORR', 'MI_PCT', 'HAS_MI', 'hurricanes_perf_window']
M2 categorical features (5): ['FIRST_FLAG', 'OCC_STAT', 'PROP', 'CHANNEL', 'NO_UNITS_GRP']TS_GRP',
]

y = df['default_180dpd'].astype(int)

print(f"M2 numeric features    ({len(numeric_features)}): {numeric_features}")
print(f"M2 categorical features ({len(categorical_features)}): {categorical_features}")


M2 numeric features    (10): ['CSCORE_B', 'DTI', 'ORIG_CLTV', 'ORIG_RATE', 'ORIG_UPB', 'ORIG_TERM', 'NUM_BORR', 'MI_PCT', 'HAS_MI', 'hurricanes_perf_window']
M2 categorical features (5): ['FIRST_FLAG', 'OCC_STAT', 'PROP', 'CHANNEL', 'NO_UNITS_GRP']


## 3. Quick sanity check on climate variable distributions

In [6]:
print("Climate variable summary (all loans):")
print(df[['hurricanes_perf_window', 'any_hurricane_perf_window']].describe().round(4))

# Cross-tab: default rate by climate exposure quintile
df['freq_quintile'] = pd.qcut(df['hurricanes_perf_window'], q=5, duplicates='drop', labels=False)
xt = df.groupby('freq_quintile').agg(
    n=('default_180dpd', 'size'),
    default_rate=('default_180dpd', 'mean'),
    mean_freq=('hurricanes_perf_window', 'mean'),   
).round(4)
print("\nDefault rate by hurricanes_perf_window quintile:")
print(xt)

Climate variable summary (all loans):
       hurricanes_perf_window  any_hurricane_perf_window
count              19625.0000                    19625.0
mean                   5.9066                        1.0
std                    0.8163                        0.0
min                    4.8841                        1.0
25%                    5.0000                        1.0
50%                    5.9333                        1.0
75%                    6.8986                        1.0
max                    7.4571                        1.0

Default rate by hurricanes_perf_window quintile:
                  n  default_rate  mean_freq
freq_quintile                               
0              6151        0.0320     4.9662
1              2488        0.0350     5.4976
2              4081        0.0247     5.9452
3              3044        0.0250     6.6316
4              3861        0.0256     7.0560


## 4. Build design matrix

In [7]:
X_num = df[numeric_features].astype(float).copy()

X_cat = pd.get_dummies(
    df[categorical_features].astype(str),
    prefix=categorical_features,
    drop_first=True,
    dtype=float,
)

X = pd.concat([X_num, X_cat], axis=1)
X = sm.add_constant(X, has_constant='add')

print(f"Design matrix shape: {X.shape}")


Design matrix shape: (19625, 21)


## 5. Stratified train/test split — SAME random_state as M1

In [8]:
# CRITICAL: use identical random_state so M1 and M2 are evaluated on the SAME test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
print(f"Train: {len(X_train):>7,}  default rate: {y_train.mean():.4f}")
print(f"Test : {len(X_test):>7,}  default rate: {y_test.mean():.4f}")


Train:  13,737  default rate: 0.0285
Test :   5,888  default rate: 0.0285


## 6. Fit M2 logit

In [24]:
m2 = sm.Logit(y_train, X_train.astype(float)).fit(disp=True, maxiter=200)
print(m2.summary())


Optimization terminated successfully.
         Current function value: 0.114395
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:         default_180dpd   No. Observations:                13737
Model:                          Logit   Df Residuals:                    13717
Method:                           MLE   Df Model:                           19
Date:                Fri, 10 Jul 2026   Pseudo R-squ.:                  0.1174
Time:                        16:00:55   Log-Likelihood:                -1571.4
converged:                       True   LL-Null:                       -1780.5
Covariance Type:            nonrobust   LLR p-value:                 7.220e-77
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                         0.8916        nan        nan        nan         

## 7. Coefficients, odds ratios, CIs

In [23]:
ci = m2.conf_int()
coef_table = pd.DataFrame({
    'coef':       m2.params,
    'std_err':    m2.bse,
    'z':          m2.tvalues,
    'p_value':    m2.pvalues,
    'odds_ratio': np.exp(m2.params),
    'or_ci_low':  np.exp(ci[0]),
    'or_ci_high': np.exp(ci[1]),
})
coef_table.to_csv(OUT_DIR / "m2_coefficients.csv")

print("Climate coefficients:")
print(coef_table.loc[['hurricanes_perf_window']].round(4))

print("\nTop 10 features by |z|:")
top = coef_table.reindex(coef_table['z'].abs().sort_values(ascending=False).index).head(10)
print(top[['coef', 'odds_ratio', 'p_value']].round(4))


Climate coefficients:
                          coef  std_err       z  p_value  odds_ratio  \
hurricanes_perf_window -0.0428   0.0737 -0.5812   0.5611      0.9581   

                        or_ci_low  or_ci_high  
hurricanes_perf_window     0.8293      1.1069  

Top 10 features by |z|:
                coef  odds_ratio  p_value
CSCORE_B     -0.0145      0.9856   0.0000
ORIG_CLTV     0.0178      1.0179   0.0000
ORIG_UPB      0.0000      1.0000   0.0000
NUM_BORR     -0.4741      0.6224   0.0000
DTI           0.0298      1.0302   0.0001
ORIG_RATE     0.3827      1.4663   0.0028
FIRST_FLAG_1  0.2428      1.2748   0.0627
OCC_STAT_P    0.6662      1.9467   0.0641
CHANNEL_C     0.1785      1.1955   0.0831
OCC_STAT_S    0.6536      1.9225   0.0971


## 8. Out-of-sample evaluation

In [11]:
p_train = m2.predict(X_train.astype(float))
p_test  = m2.predict(X_test.astype(float))

m2_metrics = pd.DataFrame({
    'split': ['train', 'test'],
    'n':     [len(y_train), len(y_test)],
    'auc':   [roc_auc_score(y_train, p_train), roc_auc_score(y_test, p_test)],
    'brier': [brier_score_loss(y_train, p_train), brier_score_loss(y_test, p_test)],
})
m2_metrics.to_csv(OUT_DIR / "m2_metrics.csv", index=False)
print("M2 evaluation:")
print(m2_metrics.round(4))


M2 evaluation:
   split      n     auc   brier
0  train  13737  0.7851  0.0267
1   test   5888  0.7677  0.0267


## 9. M1 vs M2 head-to-head comparison

This is the core dissertation finding — the incremental predictive contribution of climate exposure over traditional underwriting variables.


In [25]:
# Assemble the comparison table
m1_train_auc = m1_metrics.loc[m1_metrics['split'] == 'train', 'auc'].item()
m1_test_auc  = m1_metrics.loc[m1_metrics['split'] == 'test',  'auc'].item()
m1_train_br  = m1_metrics.loc[m1_metrics['split'] == 'train', 'brier'].item()
m1_test_br   = m1_metrics.loc[m1_metrics['split'] == 'test',  'brier'].item()

m2_train_auc = m2_metrics.loc[m2_metrics['split'] == 'train', 'auc'].item()
m2_test_auc  = m2_metrics.loc[m2_metrics['split'] == 'test',  'auc'].item()
m2_train_br  = m2_metrics.loc[m2_metrics['split'] == 'train', 'brier'].item()
m2_test_br   = m2_metrics.loc[m2_metrics['split'] == 'test',  'brier'].item()

comparison = pd.DataFrame({
    'metric':   ['AUC (train)', 'AUC (test)', 'Brier (train)', 'Brier (test)'],
    'M1':       [m1_train_auc, m1_test_auc, m1_train_br, m1_test_br],
    'M2':       [m2_train_auc, m2_test_auc, m2_train_br, m2_test_br],
    'delta':    [m2_train_auc - m1_train_auc,
                 m2_test_auc  - m1_test_auc,
                 m2_train_br  - m1_train_br,
                 m2_test_br   - m1_test_br],
})
comparison.to_csv(OUT_DIR / "m1_vs_m2_comparison.csv", index=False)
print(comparison.round(5))


          metric       M1       M2    delta
0    AUC (train)  0.78505  0.78506  0.00002
1     AUC (test)  0.76643  0.76765  0.00122
2  Brier (train)  0.02672  0.02672 -0.00000
3   Brier (test)  0.02674  0.02674 -0.00001


## 10. Likelihood ratio test — are climate variables jointly significant?

Under H0 (climate coefficients are all zero), 2 × (LL_M2 - LL_M1) follows a chi-squared distribution with df = number of added variables (= 2 here).


In [27]:
# NOTE: the LR test is only valid because M1 and M2 are nested and fit on the same training data
ll_m1 = m1_model.llf
ll_m2 = m2.llf
LR_stat = 2 * (ll_m2 - ll_m1)
df_added = 1   # hurricane_hit_in_year + hurricane_frequency_5yr
p_LR = 1 - stats.chi2.cdf(LR_stat, df=df_added)

print(f"M1 log-likelihood: {ll_m1:.3f}")
print(f"M2 log-likelihood: {ll_m2:.3f}")
print(f"LR statistic     : {LR_stat:.3f}")
print(f"df               : {df_added}")
print(f"p-value          : {p_LR:.6f}")

if p_LR < 0.001:
    print("\n=> Climate variables are JOINTLY HIGHLY SIGNIFICANT (p < 0.001)")
elif p_LR < 0.05:
    print("\n=> Climate variables are JOINTLY SIGNIFICANT (p < 0.05)")
else:
    print("\n=> Climate variables are NOT jointly significant (p >= 0.05)")


M1 log-likelihood: -1571.646
M2 log-likelihood: -1571.447
LR statistic     : 0.398
df               : 1
p-value          : 0.528169

=> Climate variables are NOT jointly significant (p >= 0.05)


## 11. Coefficient stability — did adding climate change M1 coefficients?

If ORIG_RATE's coefficient drops meaningfully in M2, this suggests that part of the rate signal in M1 was itself embedding climate risk pricing (consistent with Gete, Tsouderou & Wachter 2024's CRT evidence).


In [28]:
m1_params = m1_model.params
m2_params = m2.params

# Only compare variables that appear in both models
shared_vars = [v for v in m1_params.index if v in m2_params.index]

coef_shift = pd.DataFrame({
    'M1_coef': m1_params.loc[shared_vars],
    'M2_coef': m2_params.loc[shared_vars],
})
coef_shift['abs_shift']    = (coef_shift['M2_coef'] - coef_shift['M1_coef']).abs()
coef_shift['pct_shift']    = ((coef_shift['M2_coef'] - coef_shift['M1_coef']) / coef_shift['M1_coef'].abs()) * 100

print("Coefficient shift M1 -> M2 (sorted by absolute change):")
print(coef_shift.sort_values('abs_shift', ascending=False).round(4))


Coefficient shift M1 -> M2 (sorted by absolute change):
                    M1_coef  M2_coef  abs_shift  pct_shift
const                1.5709   0.8916     0.6792   -43.2390
HAS_MI              -0.0903  -0.0729     0.0174    19.2536
PROP_MH              0.6061   0.6207     0.0146     2.4157
PROP_PU             -0.0448  -0.0303     0.0145    32.3895
PROP_SF              0.1860   0.1980     0.0120     6.4585
NO_UNITS_GRP_2plus   0.2105   0.2011     0.0095    -4.4930
FIRST_FLAG_1         0.2503   0.2428     0.0074    -2.9757
OCC_STAT_P           0.6607   0.6662     0.0055     0.8325
CHANNEL_R            0.1096   0.1146     0.0050     4.5889
OCC_STAT_S           0.6500   0.6536     0.0036     0.5537
ORIG_RATE            0.3856   0.3827     0.0028    -0.7328
NUM_BORR            -0.4766  -0.4741     0.0025     0.5275
CHANNEL_C            0.1763   0.1785     0.0022     1.2539
MI_PCT               0.0089   0.0082     0.0007    -7.7925
ORIG_CLTV            0.0173   0.0178     0.0004     2.5483


## 12. Save fitted model

In [13]:
joblib.dump(m2, OUT_DIR / "m2_logit.pkl")
print(f"Saved M2 artifacts to {OUT_DIR}")


Saved M2 artifacts to E:\Financial Mathsmatics Master\Dissertation\datasets


In [1]:
!pip freeze > requirements.txt

## 13. Interpretation checklist

Before writing up Section 5.2, confirm:

1. **Climate coefficients significant?** — expect `hurricane_hit_in_year` and/or `hurricane_frequency_5yr` to have p < 0.05 with **positive** sign (more exposure → higher default). If negative, something's off with the climate merge.

2. **AUC lift is meaningful?** — for FL 2017 Q1, a plausible test ΔAUC from climate alone is 0.003–0.015. Small but statistically detectable is the expected story; a huge jump (>0.03) would be suspicious.

3. **LR test rejects H0?** — expect p < 0.05 given the sample size (~9,600 events × 2 climate vars). Non-rejection would be a negative finding worth reporting honestly.

4. **ORIG_RATE stability?** — if ORIG_RATE coefficient drops in M2, note it: consistent with climate being partially priced into origination rates.

5. **Cross-tab in step 3 makes sense?** — default rate should rise (weakly monotonically) across `hurricane_frequency_5yr` quintiles. If it's flat or non-monotonic, the raw signal is weak and any regression effect is being pulled out only by conditioning on covariates.
